# Evaluation -- FID, CLIP Score, Human Perference

Every generative model leaderboard cites FID, CLIP score, and a win rate from human-preference arena. Each number has a failure mode a determined researcher can game. If you do not know the failure modes, you cannot tell a real imporvement from a gaming run.

## Three Metrics

### FID (Frechet Inception Distance)-- sample quality

A distance between two distributions -- real and generated -- in an inception network's feature space. Lower is better.

Steps:
1. Extract Inception-v3 features (2048-D) for N real imanges and N generated.
2. Fit a Gaussian to each pool: compute mean `miu_r, miu_g` and covariant `coh_r, coh_g`.
3. FID = `||miu_r - miu_g||^2 + Tr(coh_r + coh_g - 2 * (coh_r * coh_g)^0.5)`

Interpretation: Frechet distance between two multivariate Gaussians in feature space. Lower = more similar distributions.

#### Failure modes

* Biased on small N.  FID is mean-squared over the feature distribution -- small N under-estimates covariance, give falsely low FID

* Inception-dependent.  Inception-v3 was trained on ImageNet. Domains far from ImageNet produce meaningless FID.

* Gaming. Overfitting to the Inception prior gives low FID without visual quality imporovement.

### CLIP score -- prmopt adherence

Cosine similarity between a generated image's CLIP-image embedding and a prompt's CLIP-text embedding. Higher is better. Measure prompt adherence.

For a generated image + prompt:
```
clip_score = cos_sim(CLIP_image(x_gen), CLIP_text(prompt))
```

#### Failure Modes
* CLIP's own blind spots.  CLIP has weak compositional reasoning. **Models can rank well on CLIP score without really following complex prompts.**

* Short prompt bias.  Short prompts have more CLIP-image matches in the wild. Longer prompts have lower CLIP scores mechanically.

* Prompt gaming. Including "hihg quality, 4k, masterpiece" in the prompt inflates CLIP score without imporving image-text binding.

### Human preference -- the ground truth
Pick a pool of prompts. Generate with model A and model B. Show pairs to humans (or a strong LLM judge)